In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os 

# Setup path untuk import dari src folder
loc = os.getcwd()
root = os.path.dirname(loc)
srcPath = os.path.join(root, "src")
sys.path.insert(0, srcPath)

from ffnn.model import FFNN
from ffnn.utils.activation_function import Linear, ReLU, Sigmoid, Tanh, Softmax
from ffnn.utils.loss_function import MSE, BinaryCrossEntropy, CategoricalCrossEntropy
from ffnn.utils.initialization import ZeroInit, UniformInit, NormalInit, XavierInit, HeInit
from ffnn.utils.regularizer import L1, L2
from ffnn.plot import plotTrainingHistory, plotWeightDistribution, plotGradientDistribution
from ffnn.utils.optimizer import GradientDescent, Adam

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.neural_network import MLPClassifier

plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)


In [16]:

# ambil dari pickle  
import pickle 
import os
loc = os.getcwd()
models  = os.path.join(loc, 'models')
with open(os.path.join(models, 'processedData.pkl'), 'rb') as f:  
    data = pickle.load(f)

# with open('processedData.pkl', 'rb') as f:  
#     data = pickle.load(f)

print(data)

Xtr = data['X_train'].values
Xte = data['X_test'].values
ytr = data['y_train'].values.reshape(-1, 1)
yte = data['y_test'].values.reshape(-1, 1)

if (Xtr is not None) and (Xte is not None) and (ytr is not None) and (yte is not None):
    print(f'Xtr shape: {Xtr.shape}')
    print(f'Xte shape: {Xte.shape}')
    print(f'ytr shape: {ytr.shape}')
    print(f'yte shape: {yte.shape}')

{'X_train':           cgpa  backlogs  internship_count  aptitude_score  \
1575  0.385195 -1.099921          1.745749        1.854680   
786   0.686205  0.679576          0.717950        0.809427   
4348 -0.045942  0.679576          1.745749       -1.609019   
8411 -0.437796  1.569325          1.745749       -0.653170   
8650 -0.396835 -0.210172         -1.337648       -0.690039   
...        ...       ...               ...             ...   
6526 -0.302160 -0.210172          0.717950       -1.134934   
922   0.545952  0.679576         -0.309849       -1.043985   
4705 -0.260639 -1.099921         -0.309849        0.218275   
3572 -2.264060  0.679576         -0.309849       -0.972783   
6512 -2.169449 -0.210172          0.717950       -1.186435   

      communication_score  internship_quality_score  college_tier_Tier 1  \
1575            -0.596811                  0.840215                  0.0   
786              1.099136                  1.591018                  0.0   
4348           

/home/andrewt/Documents/GitHub/feed-forward-neural-network/venv/lib64/python3.14/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [17]:
print(data.keys())

dict_keys(['X_train', 'X_test', 'y_train', 'y_test', 'le'])


### Define Architecture

In [18]:
layer_sizes = [Xtr.shape[1], 32, 16, 1]
activations = [ReLU(), ReLU(), Sigmoid()]
loss = BinaryCrossEntropy()
init = HeInit(seed=42)
reg = L2(0.001)

print(f'Architecture: {layer_sizes}')

Architecture: [28, 32, 16, 1]


In [ ]:
results = {}

lr_list = [0.001, 0.005, 0.01]

for lr in lr_list:
    print(f"Training with LR={lr}...")

    # GD
    model_gd = FFNN(layer_sizes, activations, loss, init, regularizer=reg)
    hist_gd = model_gd.fit(
        Xtr, ytr, Xte, yte,
        epochs=50,
        optimizer=GradientDescent(lr=lr),
        verbose=0
    )
    y_pred_gd = (model_gd.predict(Xte) > 0.5).astype(int)

    # Adam
    model_adam = FFNN(layer_sizes, activations, loss, init, regularizer=reg)
    hist_adam = model_adam.fit(
        Xtr, ytr, Xte, yte,
        epochs=50,
        optimizer=Adam(lr=lr),
        verbose=0
    )
    y_pred_adam = (model_adam.predict(Xte) > 0.5).astype(int)

    # metrics
    gd_acc = accuracy_score(yte, y_pred_gd)
    adam_acc = accuracy_score(yte, y_pred_adam)

    gd_f1 = f1_score(yte, y_pred_gd)
    adam_f1 = f1_score(yte, y_pred_adam)

    results[lr] = {
        "gd_hist": hist_gd,
        "adam_hist": hist_adam,
        "gd_acc": gd_acc,
        "adam_acc": adam_acc,
        "gd_f1": gd_f1,
        "adam_f1": adam_f1
    }

for lr in lr_list:
    r = results[lr]

    print(f"\nLR = {lr}\n")

    print(f"GD   => Loss: {r['gd_hist']['val_loss'][-1]:.4f} | Acc: {r['gd_acc']:.4f} | F1: {r['gd_f1']:.4f}")
    print(f"Adam => Loss: {r['adam_hist']['val_loss'][-1]:.4f} | Acc: {r['adam_acc']:.4f} | F1: {r['adam_f1']:.4f}")

    winner = "Adam" if r["adam_f1"] > r["gd_f1"] else "GD"
    print(f"Winner: {winner}\n")

Training with LR=0.001...


Training with LR=0.005...
Training with LR=0.01...

LR = 0.001

GD   => Acc: 0.6025 | F1: 0.7520
Adam => Acc: 0.6014 | F1: 0.7499
Winner: GD


LR = 0.005

GD   => Acc: 0.6025 | F1: 0.7520
Adam => Acc: 0.7384 | F1: 0.7980
Winner: Adam


LR = 0.01

GD   => Acc: 0.6025 | F1: 0.7520
Adam => Acc: 0.7352 | F1: 0.7921
Winner: Adam



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, lr in enumerate(lr_list):
    r = results[lr]

    axes[idx].plot(r['gd_hist']['val_loss'], label='GD', linewidth=2)
    axes[idx].plot(r['adam_hist']['val_loss'], label='Adam', linewidth=2)
    axes[idx].set_title(f'Learning Rate = {lr}')
    axes[idx].set_xlabel('Epoch')
    axes[idx].set_ylabel('Validation Loss')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('GD vs Adam: Validation Loss per Learning Rate', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


# Train loss juga

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, lr in enumerate(lr_list):
    r = results[lr]

    axes[idx].plot(r['gd_hist']['train_loss'], label='GD', linewidth=2)
    axes[idx].plot(r['adam_hist']['train_loss'], label='Adam', linewidth=2)
    axes[idx].set_title(f'Learning Rate = {lr}')
    axes[idx].set_xlabel('Epoch')
    axes[idx].set_ylabel('Training Loss')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('GD vs Adam: Training Loss per Learning Rate', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()